
# 시계열 정합성 검증 (Timeseries Consistency Check)

이 노트북은 다음 파일들의 시계열 정합성을 점검하고 기본 값 검증을 수행합니다.

- `open.csv`, `high.csv`, `low.csv`, `close.csv`, `vwap.csv`, `volume.csv`, `export_value.csv`

### 목적
1) 모든 파일이 동일한 `(date, symbol)` 축을 공유하는지 확인  
2) 파일별 누락 현황(날짜/종목)을 요약  
3) 가격 기본 규칙 위반(high/low, vwap 범위) 및 음수 거래량 탐지  
4) 요약 리포트를 `../output/timeseries_consistency_report.csv` 로 저장


In [ ]:

# === 설정 ================================================================
from pathlib import Path

# 프로젝트 구조 기준: 이 노트북이 project/src/에 위치한다고 가정
FILES = {
    "open":         "../data/price/open.csv",
    "high":         "../data/price/high.csv",
    "low":          "../data/price/low.csv",
    "close":        "../data/price/close.csv",
    "vwap":         "../data/price/vwap.csv",
    "volume":       "../data/price/volume.csv",
    "export_value": "../data/price/export_value.csv",
}

OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUTPUT_DIR / "timeseries_consistency_report.csv"

FILES, REPORT_PATH


In [ ]:

# === 라이브러리 로드 및 유틸 =============================================
import pandas as pd
import numpy as np
import os, re

def read_pivot_csv(path: str) -> pd.DataFrame:
    """
    행=날짜(yyyymmdd 또는 2020-01-03 등), 열=심볼 구조의 CSV 로드.
    - 첫 컬럼을 인덱스로 사용
    - 인덱스/컬럼 공백 제거
    - 날짜 인덱스는 20200103 형태로 정규화
    """
    df = pd.read_csv(path, index_col=0)
    df.index = df.index.map(lambda x: str(x).strip())
    df.columns = df.columns.map(lambda x: str(x).strip())
    df.index = df.index.map(lambda s: re.sub(r"[-_/\.]", "", s))
    return df

def basic_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """문자/공백 섞인 경우 float로 강제 변환(실패 시 NaN)."""
    return df.applymap(lambda x: pd.to_numeric(x, errors="coerce"))


## 1) 데이터 로드

In [ ]:

dfs = {}
for name, path in FILES.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"[{name}] 파일을 찾을 수 없습니다: {path}")
    df = read_pivot_csv(path)
    dfs[name] = df

{ k:(v.shape, v.index[:2].tolist(), v.columns[:3].tolist()) for k,v in dfs.items() }


## 2) 공통 축 및 합집합 계산

In [ ]:

date_sets = {k: set(v.index)   for k, v in dfs.items()}
sym_sets  = {k: set(v.columns) for k, v in dfs.items()}

common_dates   = set.intersection(*date_sets.values())
common_symbols = set.intersection(*sym_sets.values())

union_dates   = set.union(*date_sets.values())
union_symbols = set.union(*sym_sets.values())

len(common_dates), len(common_symbols), len(union_dates), len(union_symbols)


## 3) 파일별 누락 현황 요약

In [ ]:

rows = []
for name, df in dfs.items():
    dates = set(df.index)
    syms  = set(df.columns)

    missing_dates_vs_union = sorted(list(union_dates - dates))[:5]   # 샘플 5개만
    missing_syms_vs_union  = sorted(list(union_symbols - syms))[:5]

    rows.append({
        "file": name,
        "n_dates": len(dates),
        "n_symbols": len(syms),
        "n_missing_dates_vs_union": len(union_dates - dates),
        "n_missing_syms_vs_union": len(union_symbols - syms),
        "sample_missing_dates": ",".join(missing_dates_vs_union),
        "sample_missing_syms": ",".join(missing_syms_vs_union),
    })

summary_df = pd.DataFrame(rows).sort_values("file")
summary_df.to_csv(REPORT_PATH, index=False, encoding="utf-8-sig")
summary_df


In [ ]:

print("공통 날짜 수:", len(common_dates), "/ 합집합:", len(union_dates))
print("공통 종목 수:", len(common_symbols), "/ 합집합:", len(union_symbols))
print("요약 보고서 저장:", REPORT_PATH)


## 4) 기본 값 검증 (가격 범위, VWAP 범위, 음수 거래량)

In [ ]:

def align_common(*dfs_list):
    """여러 DF를 공통 (date, symbol) 교차영역으로 정렬."""
    idx = set.intersection(*[set(d.index) for d in dfs_list])
    cols = set.intersection(*[set(d.columns) for d in dfs_list])
    idx = sorted(idx)
    cols = sorted(cols)
    return [d.loc[idx, cols] for d in dfs_list]

# 가격 범위 체크
if all(k in dfs for k in ("open", "high", "low", "close")):
    o, h, l, c = align_common(dfs["open"], dfs["high"], dfs["low"], dfs["close"])
    o, h, l, c = map(basic_numeric, (o, h, l, c))

    cond_high = (h >= o) & (h >= c)
    cond_low  = (l <= o) & (l <= c)
    bad_high = (~cond_high) & (~h.isna() & ~o.isna() & ~c.isna())
    bad_low  = (~cond_low)  & (~l.isna() & ~o.isna() & ~c.isna())

    n_bad_high = int(bad_high.sum().sum())
    n_bad_low  = int(bad_low.sum().sum())
    print(f"[값 검증] high 범위 위반 개수: {n_bad_high}, low 범위 위반 개수: {n_bad_low}")
else:
    print("open/high/low/close 중 일부가 없어 가격 범위 검증을 생략합니다.")

# vwap 범위 체크 (가능한 경우)
if all(k in dfs for k in ("vwap", "high", "low")):
    v, h, l = align_common(dfs["vwap"], dfs["high"], dfs["low"])
    v, h, l = map(basic_numeric, (v, h, l))
    cond_vwap = (v <= h) & (v >= l)
    bad_vwap = (~cond_vwap) & (~v.isna() & ~h.isna() & ~l.isna())
    n_bad_vwap = int(bad_vwap.sum().sum())
    print(f"[값 검증] vwap ∈ [low, high] 위반 개수: {n_bad_vwap}")
else:
    print("vwap/high/low 중 일부가 없어 vwap 범위 검증을 생략합니다.")

# volume 음수 체크
if "volume" in dfs:
    vol = basic_numeric(dfs["volume"])
    n_neg_vol = int((vol < 0).sum().sum())
    print(f"[값 검증] 음수 거래량 개수: {n_neg_vol}")
else:
    print("volume 파일이 없어 음수 거래량 검증을 생략합니다.")



## 5) 다음 단계
- 위 요약 결과를 바탕으로 결측치 처리 기준을 결정한다.
  - 자연 결측(휴장일 등)은 전체 날짜 drop
  - 비정상 결측(특정 종목만 결측)은 적절한 보간(예: forward fill) 또는 해당 종목 제외
- 결측 처리 일괄 로직을 별도 노트북 또는 본 노트북 후반부에 구현한다.
